In [1]:
# Housekeeping, imports
import pandas as pd
import numpy as np
import os, re, glob
from IPython.display import Markdown

In [2]:
# Create folders if needed
if not(os.path.exists('mapping')):
   os.mkdir('mapping')
for format in ['csv', 'md', 'json', 'xml', 'excel']:
    if not(os.path.exists(os.path.join('mapping', format))):
       os.mkdir(os.path.join('mapping', format))

In [3]:
# Housekeeping, imports
zibinfo = pd.read_excel('deprecated_zib_versions.xlsx', sheet_name='table_info_zibs')
zibinfo = zibinfo.fillna('')
zibinfo.head()

,zib,wiki_URL_2020,wiki_URL_2017,OID_zib2020,OID_zib2017,sub_header_1,sub_header_2
0,Behandelaanwijzing,https://zibs.nl/wiki/BehandelAanwijzing2-v1.0(...,https://zibs.nl/wiki/BehandelAanwijzing-v3.1(2...,2.16.840.1.113883.2.4.3.11.60.40.3.2.2,2.16.840.1.113883.2.4.3.11.60.40.3.2.1,,


In [4]:
excels = os.listdir("mapping/excel/")
excels = [excel for excel in excels if excel.endswith(".xlsx")]
dfs = []
for excel in excels:
    dfs.append(pd.read_excel("mapping/excel/" + excel))
df = pd.concat(dfs)
df.head()

,ZibName 2017,ConceptID_2017,ConceptName_2017,Codelists_2017,Relation,Zibname 2020,ConceptID_2020,ConceptName_2020,Codelists_2020,TRANSLATIE_2017_naar_2020,TRANSLATIE_2020_naar_2017,Opmerkingen
0,BehandelAanwijzing,NL-CM:2.1.1,BehandelAanwijzing,NaN,is equivalent to,BehandelAanwijzing2,NL-CM:2.2.1,BehandelAanwijzing2,NaN,NaN,NaN,NaN
1,BehandelAanwijzing,NL-CM:2.1.8,Verificatie,NaN,NaN,NaN,NaN,**concept verwijderd in 2020**,NaN,Zie stroomdiagram Verificatie,nvt,NaN
2,BehandelAanwijzing,NL-CM:2.1.9,Geverifieerd,NaN,NaN,NaN,NaN,**concept verwijderd in 2020**,NaN,NaN,nvt,NaN
3,BehandelAanwijzing,NL-CM:2.1.11,GeverifieerdBij,GeverifieerdBijCodelijst,is equivalent to,BehandelAanwijzing2,NL-CM:2.2.9,AfspraakPartij,NaN,Als GeverifieerdBij == Patient dan AfspraakPar...,Als AfspraakPartij == verwijzing naar Patient ...,NaN
4,BehandelAanwijzing,NL-CM:2.1.10,VerificatieDatum,NaN,is equivalent to,BehandelAanwijzing2,NL-CM:2.2.5,MeestRecenteBespreekdatum,NaN,MeestRecenteBespreekdatum = VerificatieDatum. ...,VerificatieDatum = MeestRecenteBespreekdatum,NaN


In [5]:
# Replace nulls with emtpy string
df = df.fillna('')
cols = df.columns
cols

Index(['ZibName 2017', 'ConceptID_2017', 'ConceptName_2017', 'Codelists_2017',
       'Relation', 'Zibname 2020', 'ConceptID_2020', 'ConceptName_2020',
       'Codelists_2020', 'TRANSLATIE_2017_naar_2020',
       'TRANSLATIE_2020_naar_2017', 'Opmerkingen'],
      dtype='object')

In [ ]:
# --- Version information (read from the first Excel file that has the sheet) ---
versie = ''
volwassenheidsniveau = ''

for excel in excels:
    try:
        vi_df = pd.read_excel(excel, sheet_name='version_information').fillna('')
        # Try direct columns named "Versie" / "Volwassenheidsniveau"
        versie_cols = [c for c in vi_df.columns if str(c).strip().casefold() == 'versie']
        volw_cols  = [c for c in vi_df.columns if str(c).strip().casefold() == 'volwassenheidsniveau']
        if versie_cols:
            vals = [str(x).strip() for x in vi_df[versie_cols[0]].tolist() if str(x).strip() != '']
            if vals: versie = vals[0]
        if volw_cols:
            vals = [str(x).strip() for x in vi_df[volw_cols[0]].tolist() if str(x).strip() != '']
            if vals: volwassenheidsniveau = vals[0]

        # Fallback for key/value-style sheets (e.g. Naam/Waarde)
        if versie == '' or volwassenheidsniveau == '':
            for kc in vi_df.columns:
                for vc in vi_df.columns:
                    if vc == kc:
                        continue
                    # Versie
                    if versie == '':
                        r = vi_df[vi_df[kc].astype(str).str.strip().str.casefold() == 'versie']
                        if not r.empty:
                            cell = str(r.iloc[0][vc]).strip()
                            if cell: versie = cell
                    # Volwassenheidsniveau
                    if volwassenheidsniveau == '':
                        r = vi_df[vi_df[kc].astype(str).str.strip().str.casefold() == 'volwassenheidsniveau']
                        if not r.empty:
                            cell = str(r.iloc[0][vc]).strip()
                            if cell: volwassenheidsniveau = cell

        # Last fallback: scan for a cell with label and take the cell to the right
        if versie == '' or volwassenheidsniveau == '':
            for i in range(len(vi_df)):
                row = vi_df.iloc[i]
                for j in range(len(vi_df.columns) - 1):
                    lab = str(row.iloc[j]).strip().casefold()
                    if versie == '' and lab == 'versie':
                        versie = str(row.iloc[j+1]).strip()
                    if volwassenheidsniveau == '' and lab == 'volwassenheidsniveau':
                        volwassenheidsniveau = str(row.iloc[j+1]).strip()
        break  # we found and processed version_information; stop checking other files
    except Exception:
        continue

# Escape for Markdown cells
versie_md = str(versie).replace('|', r'\|').replace('\n', '<br>')
volw_md   = str(volwassenheidsniveau).replace('|', r'\|').replace('\n', '<br>')

# Prebuilt section to prepend to every MD file
version_section = (
    "## Version information\n\n"
    "|  |  |\n"
    "|---|---|\n"
    f"| Versie | {versie_md} |\n"
    f"| Volwassenheidsniveau | {volw_md} |\n\n"
)

In [6]:
# Make a list of zibs.
zibs = sorted([z for z in df['ZibName 2017'].dropna().unique() if str(z).strip()])
zibs

['BehandelAanwijzing']

In [ ]:
# 5) Loop through zibs
for zib in zibs:
    if len(str(zib).strip()) == 0:
        continue

    tdf = df[df['ZibName 2017'] == zib].copy().fillna('')

    # --- Subheader (zibinfo optional) ---
    subheader_parts = []
    if 'zibinfo' in globals():
        try:
            zrow = zibinfo[zibinfo['zib'] == zib]
            if not zrow.empty:
                for col in ('wiki_URL_2017', 'wiki_URL_2020'):
                    if col in zrow.columns:
                        url = str(zrow.iloc[0][col])
                        if url and url != 'nan':
                            tail = url.rstrip('/').split('/')[-1]
                            subheader_parts.append(f"[{tail}]({url})")
                for col in ('sub_header_1', 'sub_header_2'):
                    if col in zrow.columns:
                        txt = str(zrow.iloc[0][col])
                        if txt and txt != 'nan':
                            subheader_parts.append(txt)
        except Exception:
            pass
    subheader = "\n\n".join(subheader_parts)

    # --- Detect ZIB-level rows (robust rule) ---
    name_match = tdf['ConceptName_2017'].str.casefold() == str(zib).casefold()
    id_match = tdf['ConceptID_2017'].str.match(r'^NL-CM:\d+\.\d+\.1(?:$|\.)', na=False)
    hmask = name_match | id_match

    # --- Escape values for Markdown ---
    tdf = tdf.applymap(lambda x: str(x).replace('|', '\\|').replace('\n', '<br>'))

    smallcols = ['ConceptID_2017','ConceptName_2017','ConceptID_2020','ConceptName_2020','Relation']

    # --- ZIB-level changes ---
    htable = tdf.loc[hmask, smallcols].sort_values(['ConceptID_2017','ConceptName_2017'])
    mdhtable = '## Zib-level changes\n\n' + htable.to_markdown(index=False) if not htable.empty else ''

    # --- Changes section ---
    changemask = (~hmask) & \
                 ((tdf.get('TRANSLATIE_2017_naar_2020','')!='') | 
                  (tdf.get('TRANSLATIE_2020_naar_2017','')!=''))
    changes_df = tdf.loc[changemask, smallcols].sort_values(['ConceptID_2017','ConceptName_2017'])
    mdchanges = changes_df.to_markdown(index=False) if not changes_df.empty else '_No changes detected_'

    # --- Full mapping ---
    mapping_df = tdf.loc[~hmask, :].sort_values(['ConceptID_2017','ConceptName_2017'])
    mdmapping = mapping_df.loc[:, cols].to_markdown(index=False)

    # --- File header ---
    slug = re.sub(r'[^A-Za-z0-9._-]+', '-', str(zib)).strip('-')
    header = f"# {zib}\n## File formats\n\nThe translation specs are available as: \n[CSV](../csv/{slug}.csv) [JSON](../json/{slug}.json) [XML](../xml/{slug}.xml)\n\n"

    content = version_section + header + (subheader + "\n\n" if subheader else "\n") + mdhtable + \
              "\n\n## Changes\n\n" + mdchanges + "\n\n## Mapping\n\n" + mdmapping + "\n\n"

    out_path = os.path.join('mapping', 'md', f'{slug}.md')
    with open(out_path, 'w', encoding='utf8') as f:
        f.write(content)

C:\Users\zanen_nict1\AppData\Local\Temp\ipykernel_29220\808323560.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  tdf = tdf.applymap(lambda x: str(x).replace('|', '\\|').replace('\n', '<br>'))


In [13]:
for zib in zibs:
    if len(str(zib).strip()) == 0:
        continue

    tdf = df[df['ZibName 2017'] == zib]

    # veilige bestandsnaam
    slug = re.sub(r'[^A-Za-z0-9._-]+', '-', str(zib)).strip('-')

    # CSV & JSON kunnen met de originele kolomnamen
    tdf.to_csv(os.path.join('mapping', 'csv',  f'{slug}.csv'), index=False)
    tdf.to_json(os.path.join('mapping', 'json', f'{slug}.json'), orient='records')

    # --- XML: maak een kopie met geldige XML-tag-namen ---
    tdf_xml = tdf.copy()
    col_map = {}
    for c in tdf_xml.columns:
        s = re.sub(r'[^A-Za-z0-9_.-]', '_', str(c))  # spaties/specials -> _
        if not re.match(r'^[A-Za-z_]', s):           # mag niet met cijfer beginnen
            s = 'n_' + s
        if s.lower().startswith('xml'):              # vermijd xml- prefix
            s = 'n_' + s
        col_map[c] = s
    tdf_xml = tdf_xml.rename(columns=col_map)

    # schijf XML weg met duidelijke root/row-namen
    tdf_xml.to_xml(os.path.join('mapping', 'xml', f'{slug}.xml'),
                   index=False, root_name='table', row_name='row')

In [14]:
redcolor = {"bg_color": "#FFC7CE"}
orangecolor = {"bg_color": "#FFA003"}
yellowcolor = {"bg_color": "#FFE403"}
greencolor = {"bg_color": "#C6EFCE"}

In [ ]:
# Only run this cell to regenerate all indivudual zib Excel files, only needed if there are format(ting) changes
# for zib in zibs:
#     if len(zib) == 0:
#         continue
#     tdf = df[df['ZibName'] == zib]
#     writer = pd.ExcelWriter(os.path.join('mapping', 'excel', zib + '.xlsx'), engine='xlsxwriter')
#     (max_row, max_col) = tdf.shape
#     workbook = writer.book
#     red = workbook.add_format(redcolor)
#     orange = workbook.add_format(orangecolor)
#     yellow = workbook.add_format(yellowcolor)
#     green = workbook.add_format(greencolor)
#     tdf.to_excel(writer, index=False, sheet_name='translations')
#     worksheet = writer.sheets['translations']
#     worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"groen: geen wijzigingen"', "format": green})
#     worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"geel: patch wijziging"', "format": yellow})
#     worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"oranje: minor change"', "format": orange})
#     worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"rood: major change"', "format": red})
#     worksheet.set_column('A:O', 40)
#     worksheet.set_column('B:B', 15)
#     worksheet.set_column('F:F', 15)
#     worksheet.autofilter(0, 0, max_row, max_col - 1)
#     writer.close()

In [15]:
df.to_csv(os.path.join('all-translations.csv'), index=False)

In [16]:
from datetime import datetime

now = datetime.now()
formatted_date = now.strftime("%d-%m-%Y %H:%M")

writer = pd.ExcelWriter('all-translations.xlsx', engine='xlsxwriter')
(max_row, max_col) = df.shape
workbook = writer.book
about = workbook.add_worksheet('About')
about.write(1, 1, 'Generated file. Do not edit this workbook, edit the individual zib workbooks instead.')
about.write(1, 2, formatted_date)
red = workbook.add_format(redcolor)
orange = workbook.add_format(orangecolor)
yellow = workbook.add_format(yellowcolor)
green = workbook.add_format(greencolor)
df.to_excel(writer, index=False, sheet_name='translations')
worksheet = writer.sheets['translations']
worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"groen: geen wijzigingen"', "format": green})
worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"geel: patch wijziging"', "format": yellow})
worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"oranje: minor change"', "format": orange})
worksheet.conditional_format(1, 4, max_row, 4, {"type": "cell", "criteria": "equal to", "value": '"rood: major change"', "format": red})
worksheet.freeze_panes(1, 1)
worksheet.set_column('A:O', 40)
worksheet.set_column('B:B', 15)
worksheet.set_column('F:F', 15)
worksheet.autofilter(0, 0, max_row, max_col - 1)
writer.close()

In [17]:
mdindex = "# Index of available zib translations\n\n"
for zib in sorted(zibs):
    mdindex = mdindex + "* [ " + zib + "](mapping/md/" + zib + ".md)\n"
with open('index.md', 'w', encoding='utf8') as file:
    file.write(mdindex)